# Section 2.2 - On-Time Arrival Estimator

This model predicts total trip time in minutes before pickup. `dropoff_timestamp`, actual `distance_miles`, and `speed_mph` are never used as predictors.

**Deployment assumption:** the destination zone is entered before the estimate is requested. Therefore `dest_loc_id` and destination zone features are available at prediction time.

**Model progression:** global median -> training-only OD median -> LightGBM gradient-boosted tree regression.

**Evaluation:** chronological train, validation, and out-of-time test periods. No random split is used.

This notebook uses Version A, a dataset-only estimator based on zones, pickup timing, calendar patterns, and booking/dispatch fields. A routing service can be added later to provide a separate pre-trip route-distance feature.

In [1]:
import os
import sys
import pandas as pd

sys.path.insert(0, os.path.abspath('..'))
from src.models.arrival_model import (
    ARRIVAL_FEATURES,
    ArrivalEstimator,
    build_historical_duration_stats,
    chronological_split,
    evaluate_arrival_predictions,
    prepare_training_frame,
)

print('Arrival estimator modules loaded.')

Arrival estimator modules loaded.


## 1. Load the model components

Import the leakage-safe feature builder, chronological splitter, estimator, and evaluation helpers.

In [4]:
import pyarrow.parquet as pq

source_path = '../data/processed/Urban_Flow_Analytics_Taxi_Trip_Time.parquet'
if not os.path.exists(source_path):
    source_path = '../data/processed/Urban_Flow_Analytics_Taxi_Clean_Enriched_12Month.parquet'

columns = [
    'pickup_timestamp', 'dropoff_timestamp',
    'provider_code', 'rider_count', 'rate_class_id',
    'origin_loc_id', 'dest_loc_id',
    'is_cross_borough', 'is_airport_trip',
    'origin_borough', 'dest_borough',
    'origin_service_zone', 'dest_service_zone',
    'origin_zone', 'dest_zone',
]
available = pq.ParquetFile(source_path).schema.names
columns = [column for column in columns if column in available]

max_rows_per_split = 500_000
split_filters = [
    [('pickup_timestamp', '<', pd.Timestamp('2025-12-01'))],
    [
        ('pickup_timestamp', '>=', pd.Timestamp('2025-12-01')),
        ('pickup_timestamp', '<', pd.Timestamp('2026-02-01')),
    ],
    [('pickup_timestamp', '>=', pd.Timestamp('2026-02-01'))],
]
split_frames = []
for filters in split_filters:
    split = pd.read_parquet(
        source_path,
        columns=columns,
        filters=filters,
        engine='pyarrow',
    )
    split = split.sort_values('pickup_timestamp').head(max_rows_per_split).copy()
    split['od_pair'] = (
        split['origin_loc_id'].astype('string')
        + '_'
        + split['dest_loc_id'].astype('string')
    )
    split_frames.append(split)

train_raw, validation_raw, test_raw = split_frames
print('Chronological sample rows:', len(train_raw), len(validation_raw), len(test_raw))

Chronological sample rows: 500000 500000 500000


## 2. Load chronological trip samples

Read only bounded train, validation, and out-of-time test samples from Parquet to avoid memory errors. The destination is assumed to be known before departure.

In [5]:
historical_stats = build_historical_duration_stats(
    train_raw,
    max_duration_minutes=180,
)
X_train, y_train = prepare_training_frame(
    train_raw,
    max_duration_minutes=180,
    historical_stats=historical_stats,
)
X_validation, y_validation = prepare_training_frame(
    validation_raw,
    max_duration_minutes=180,
    historical_stats=historical_stats,
)
X_test, y_test = prepare_training_frame(
    test_raw,
    max_duration_minutes=180,
    historical_stats=historical_stats,
)

assert set(ARRIVAL_FEATURES).isdisjoint({
    'distance_miles', 'dropoff_timestamp', 'trip_duration_seconds',
    'trip_duration_minutes', 'trip_duration_hours', 'speed_mph',
    'charge_total', 'driver_tip_payment', 'toll_total',
    'fare_settlement_method',
})

baseline = y_train.median()
baseline_metrics = evaluate_arrival_predictions(
    y_validation, [baseline] * len(y_validation)
)
estimator = ArrivalEstimator.fit(
    X_train,
    y_train,
    X_validation,
    y_validation,
    historical_stats=historical_stats,
)
validation_metrics = evaluate_arrival_predictions(
    y_validation, estimator.predict_minutes(validation_raw.loc[X_validation.index])
)
test_metrics = evaluate_arrival_predictions(
    y_test, estimator.predict_minutes(test_raw.loc[X_test.index])
)

print('Training-only historical lookup features: od, od+hour, od+weekday')
print('Median baseline:', baseline_metrics)
print('Validation:', validation_metrics)
print('Out-of-time test:', test_metrics)

C:\Users\sanuj\AppData\Roaming\Python\Python312\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training-only historical lookup features: od, od+hour, od+weekday
Median baseline: {'MAE_minutes': 11.542725563049316, 'RMSE_minutes': 18.737258854612797, 'R2': -0.17765378952026367, 'within_2_minutes_pct': 15.451330869242422, 'within_5_minutes_pct': 37.81525721867239, 'within_10_minutes_pct': 67.45759048730166}
Validation: {'MAE_minutes': 7.263379187392984, 'RMSE_minutes': 13.487179813854357, 'R2': 0.3898343127768955, 'within_2_minutes_pct': 31.879426308406263, 'within_5_minutes_pct': 61.35006033487356, 'within_10_minutes_pct': 80.18234539563868}
Out-of-time test: {'MAE_minutes': 6.885690330470237, 'RMSE_minutes': 12.735160413018715, 'R2': 0.2842083088298385, 'within_2_minutes_pct': 31.10612543880365, 'within_5_minutes_pct': 61.266796623263296, 'within_10_minutes_pct': 81.48218567248536}


## 3. Build features and train the estimator

Create the target from pickup-to-dropoff time, calculate historical traffic statistics from training data only, compare the median baseline, and train LightGBM without post-trip predictors.

In [6]:
from src.models.arrival_model import evaluate_arrival_segments

test_predictions = estimator.predict_minutes(test_raw.loc[X_test.index])
metrics_table = pd.DataFrame([evaluate_arrival_predictions(y_test, test_predictions)])
segment_table = evaluate_arrival_segments(
    test_raw.loc[X_test.index],
    y_test,
    test_predictions,
)

display(metrics_table)
display(segment_table)

example = test_raw.loc[X_test.index].iloc[[0]].copy()
example['predicted_duration_minutes'] = test_predictions[0]
example['estimated_arrival_timestamp'] = estimator.predict_arrival(example).iloc[0]
display(example[['pickup_timestamp', 'predicted_duration_minutes', 'estimated_arrival_timestamp']])

,MAE_minutes,RMSE_minutes,R2,within_2_minutes_pct,within_5_minutes_pct,within_10_minutes_pct
0,6.88569,12.73516,0.284208,31.106125,61.266797,81.482186


,segment,rows,MAE_minutes,RMSE_minutes,R2,within_2_minutes_pct,within_5_minutes_pct,within_10_minutes_pct
0,overall,499654,6.885690,12.735160,0.284208,31.106125,61.266797,81.482186
1,rush_hour,217190,7.395790,13.774807,0.267046,29.663428,59.398683,80.264745
2,non_rush_hour,282464,6.493468,11.874019,0.297337,32.215433,62.703212,82.418290
3,weekday,378125,7.382736,13.813990,0.259765,30.845355,60.064529,79.863008
4,weekend,121529,5.339183,8.547962,0.360306,31.917485,65.007529,86.520090
5,short_trip_0_15m,249695,2.675905,3.610299,0.001740,48.094275,86.300487,98.785318
6,medium_trip_15_45m,222430,8.162569,10.524803,-0.987022,15.332015,39.455109,69.703277
7,long_trip_over_45m,27529,34.752533,43.936553,-4.011897,4.471648,10.439900,19.710124


,pickup_timestamp,predicted_duration_minutes,estimated_arrival_timestamp
2336061,2026-02-01,13.250902,2026-02-01 00:13:15.054109608


## 4. Evaluate model performance

Measure MAE, RMSE, prediction accuracy within practical minute bands, and performance across traffic periods, trip lengths, weekdays, weekends, and boroughs.

In [7]:
estimator.save('../models/arrival_time_estimator.pkl')
request = test_raw.iloc[[0]].copy()
request['pickup_timestamp'] = pd.to_datetime(request['pickup_timestamp'])
request['estimated_trip_minutes'] = estimator.predict_minutes(request)
request['estimated_arrival_timestamp'] = estimator.predict_arrival(request).to_numpy()
display(request[['pickup_timestamp', 'estimated_trip_minutes', 'estimated_arrival_timestamp']])

,pickup_timestamp,estimated_trip_minutes,estimated_arrival_timestamp
2336061,2026-02-01,13.250902,2026-02-01 00:13:15.054109608


## 5. Save the model and generate a passenger ETA

Save the trained estimator and convert predicted trip minutes into an estimated arrival timestamp.